# Test LLM Feature Generation

This notebook tests the LLM feature generation functions on a small sample of players so you can verify the quality and reasoning before running on all 71 players.


In [18]:
import anthropic
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key)

## Manual API Key Setup (if needed)

If the above cell didn't find your API key, uncomment and run the cell below.


In [19]:
def test_press_rating(player_name, week, year=2025):
    """Test press rating with full response details"""
    prompt = f"""
    Search for recent news and sentiment about NFL running back {player_name}
    around week {week} of the {year} season.

    Based on the press coverage, rate the player's public perception on a scale of 1-10:
    - 1-3: Negative coverage (injury concerns, poor performance, controversy)
    - 4-6: Neutral or mixed coverage
    - 7-10: Positive coverage (breakout performance, healthy, favorable matchup)

    First, briefly explain what you found in the search results (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
        tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
    )

    print(f"\n{'='*80}")
    print(f"PRESS RATING: {player_name} - Week {week}")
    print(f"{'='*80}")

    # Show all content blocks
    for i, block in enumerate(response.content):
        if block.type == "text":
            print(f"\n[Text Block {i}]")
            print(block.text)
        elif block.type == "server_tool_use":
            print(f"\n[Search Query {i}]")
            print(f"Query: {block.input.get('query', 'N/A')}")
        elif block.type == "web_search_tool_result":
            print(f"\n[Search Results {i}]")
            for result in block.content:
                if hasattr(result, "url"):
                    print(f"  - {result.title}")
                    print(f"    URL: {result.url}")

    # Extract final rating
    text_content = ""
    for block in response.content:
        if block.type == "text":
            text_content += block.text

    # Try to extract the number from the last line
    lines = text_content.strip().split("\n")
    rating = None
    for line in reversed(lines):
        try:
            rating = int(line.strip())
            if 1 <= rating <= 10:
                break
        except:
            continue

    print(f"\n{'='*80}")
    print(f"FINAL RATING: {rating if rating else 'PARSE ERROR'}")
    print(f"{'='*80}")

    return rating if rating else 5


def test_injury_concern(player_name, week, year=2025):
    """Test injury concern with full response details"""
    prompt = f"""
    Search for injury reports about NFL running back {player_name}
    around week {week} of the {year} season.

    Rate the injury concern level on a scale of 1-5:
    - 1: No injury concerns, fully healthy
    - 2: Minor issue, questionable but likely to play
    - 3: Moderate concern, may be limited
    - 4: Significant concern, doubtful to play
    - 5: Out or ruled out

    First, briefly explain what you found in injury reports (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 5.
    """

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
        tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
    )

    print(f"\n{'='*80}")
    print(f"INJURY CONCERN: {player_name} - Week {week}")
    print(f"{'='*80}")

    for i, block in enumerate(response.content):
        if block.type == "text":
            print(f"\n[Text Block {i}]")
            print(block.text)
        elif block.type == "server_tool_use":
            print(f"\n[Search Query {i}]")
            print(f"Query: {block.input.get('query', 'N/A')}")
        elif block.type == "web_search_tool_result":
            print(f"\n[Search Results {i}]")
            for result in block.content:
                if hasattr(result, "url"):
                    print(f"  - {result.title}")
                    print(f"    URL: {result.url}")

    text_content = ""
    for block in response.content:
        if block.type == "text":
            text_content += block.text

    lines = text_content.strip().split("\n")
    rating = None
    for line in reversed(lines):
        try:
            rating = int(line.strip())
            if 1 <= rating <= 5:
                break
        except:
            continue

    print(f"\n{'='*80}")
    print(f"FINAL RATING: {rating if rating else 'PARSE ERROR'}")
    print(f"{'='*80}")

    return rating if rating else 1


def test_opponent_defense(opponent_team, week, year=2025):
    """Test opponent defense rating with full response details"""
    prompt = f"""
    Search for information about the {opponent_team} run defense 
    around week {week} of the {year} NFL season.

    Rate their run defense strength on a scale of 1-10:
    - 1-3: Elite run defense (top ranked, healthy, tough matchup for RBs)
    - 4-6: Average run defense
    - 7-10: Weak run defense (injuries, poor ranking, favorable for RBs)

    First, briefly explain what you found about their run defense (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
        tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
    )

    print(f"\n{'='*80}")
    print(f"OPPONENT DEFENSE: {opponent_team} - Week {week}")
    print(f"{'='*80}")

    for i, block in enumerate(response.content):
        if block.type == "text":
            print(f"\n[Text Block {i}]")
            print(block.text)
        elif block.type == "server_tool_use":
            print(f"\n[Search Query {i}]")
            print(f"Query: {block.input.get('query', 'N/A')}")
        elif block.type == "web_search_tool_result":
            print(f"\n[Search Results {i}]")
            for result in block.content:
                if hasattr(result, "url"):
                    print(f"  - {result.title}")
                    print(f"    URL: {result.url}")

    text_content = ""
    for block in response.content:
        if block.type == "text":
            text_content += block.text

    lines = text_content.strip().split("\n")
    rating = None
    for line in reversed(lines):
        try:
            rating = int(line.strip())
            if 1 <= rating <= 10:
                break
        except:
            continue

    print(f"\n{'='*80}")
    print(f"FINAL RATING: {rating if rating else 'PARSE ERROR'}")
    print(f"{'='*80}")

    return rating if rating else 5


def test_vegas_sentiment(player_name, week, year=2025):
    """Test vegas sentiment with full response details"""
    prompt = f"""
    Search for betting lines, prop lines, and expert picks for NFL running back {player_name}
    for week {week} of the {year} season.

    Rate the Vegas/expert sentiment on a scale of 1-10:
    - 1-3: Bearish - low prop lines, experts fading, unfavorable odds
    - 4-6: Neutral - average expectations
    - 7-10: Bullish - high prop lines, experts hyping, favorable odds

    First, briefly explain what you found about betting lines and expert picks (2-3 sentences).
    Then on a new line, provide ONLY a single number between 1 and 10.
    """

    response = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
        tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
    )

    print(f"\n{'='*80}")
    print(f"VEGAS SENTIMENT: {player_name} - Week {week}")
    print(f"{'='*80}")

    for i, block in enumerate(response.content):
        if block.type == "text":
            print(f"\n[Text Block {i}]")
            print(block.text)
        elif block.type == "server_tool_use":
            print(f"\n[Search Query {i}]")
            print(f"Query: {block.input.get('query', 'N/A')}")
        elif block.type == "web_search_tool_result":
            print(f"\n[Search Results {i}]")
            for result in block.content:
                if hasattr(result, "url"):
                    print(f"  - {result.title}")
                    print(f"    URL: {result.url}")

    text_content = ""
    for block in response.content:
        if block.type == "text":
            text_content += block.text

    lines = text_content.strip().split("\n")
    rating = None
    for line in reversed(lines):
        try:
            rating = int(line.strip())
            if 1 <= rating <= 10:
                break
        except:
            continue

    print(f"\n{'='*80}")
    print(f"FINAL RATING: {rating if rating else 'PARSE ERROR'}")
    print(f"{'='*80}")

    return rating if rating else 5

## Test Cases

Testing with 3 well-known players from different scenarios:

- **Saquon Barkley Week 5**: Elite RB, likely positive coverage
- **Derrick Henry Week 5**: Veteran RB, playoff implications
- **Josh Jacobs Week 5**: Mid-season performance


In [22]:
# Test Case 1: Saquon Barkley Week 5 vs DAL
print("\n" + "#" * 80)
print("TEST CASE 1: Saquon Barkley - Week 5, 2025")
print("#" * 80)

press = test_press_rating("Saquon Barkley", 5, 2025)
injury = test_injury_concern("Saquon Barkley", 5, 2025)
opp_def = test_opponent_defense("DAL", 5, 2025)
vegas = test_vegas_sentiment("Saquon Barkley", 5, 2025)

print(f"\n\nSUMMARY - Saquon Barkley Week 5:")
print(f"  Press Rating: {press}/10")
print(f"  Injury Concern: {injury}/5")
print(f"  Opponent Defense (DAL): {opp_def}/10")
print(f"  Vegas Sentiment: {vegas}/10")


################################################################################
TEST CASE 1: Saquon Barkley - Week 5, 2025
################################################################################

PRESS RATING: Saquon Barkley - Week 5

[Search Query 0]
Query: Saquon Barkley NFL week 5 2025 news

[Search Results 1]
  - Saquon Barkley Fantasy Week 5: Projections vs. Broncos, Points and Stats, Start or Sit - Bleacher Nation
    URL: https://www.bleachernation.com/picks/2025/10/01/saquon-barkley-fantasy-week-5-projections-vs-broncos-points-and-stats-start-or-sit/
  - Denver Broncos vs. Philadelphia Eagles updated prediction featuring J.K. Dobbins, Saquon Barkley [Week 5, 2025]
    URL: https://www.dimers.com/news/denver-broncos-vs-philadelphia-eagles-midweek-prediction-nfl-week-5-2025-ac
  - Saquon Barkley - Philadelphia Eagles Running Back - ESPN
    URL: https://www.espn.com/nfl/player/_/id/3929630/saquon-barkley
  - NFL picks: Saquon Barkley part of Week 5 running backs parlay

In [23]:
# Test Case 2: Derrick Henry Week 15 vs PIT
print("\n" + "#" * 80)
print("TEST CASE 2: Derrick Henry - Week 15, 2024")
print("#" * 80)

press = test_press_rating("Derrick Henry", 15, 2024)
injury = test_injury_concern("Derrick Henry", 15, 2024)
opp_def = test_opponent_defense("PIT", 15, 2024)
vegas = test_vegas_sentiment("Derrick Henry", 15, 2024)

print(f"\n\nSUMMARY - Derrick Henry Week 15:")
print(f"  Press Rating: {press}/10")
print(f"  Injury Concern: {injury}/5")
print(f"  Opponent Defense (PIT): {opp_def}/10")
print(f"  Vegas Sentiment: {vegas}/10")


################################################################################
TEST CASE 2: Derrick Henry - Week 15, 2024
################################################################################

PRESS RATING: Derrick Henry - Week 15

[Search Query 0]
Query: Derrick Henry week 15 2024 NFL season news

[Search Results 1]
  - Derrick Henry - Baltimore Ravens Running Back - ESPN
    URL: https://www.espn.com/nfl/player/_/id/3043078/derrick-henry
  - Derrick Henry Fantasy Football News, Rankings, Projections | Baltimore Ravens | FantasyPros
    URL: https://www.fantasypros.com/nfl/players/derrick-henry.php
  - Derrick Henry Week 15 Outlook for Fantasy Football (2024)
    URL: https://www.rotoballer.com/player-news/derrick-henry-draws-a-great-matchup-in-week-15/1516993
  - Derrick Henry 2024 Game Log | Pro-Football-Reference.com
    URL: https://www.pro-football-reference.com/players/H/HenrDe00/gamelog/2024/
  - Ravens RB Derrick Henry 'still pissed' after third fumble in three w

In [24]:
# Test Case 3: Josh Jacobs Week 5 vs LAR
print("\n" + "#" * 80)
print("TEST CASE 3: Josh Jacobs - Week 5, 2024")
print("#" * 80)

press = test_press_rating("Josh Jacobs", 5, 2024)
injury = test_injury_concern("Josh Jacobs", 5, 2024)
opp_def = test_opponent_defense("LAR", 5, 2024)
vegas = test_vegas_sentiment("Josh Jacobs", 5, 2024)

print(f"\n\nSUMMARY - Josh Jacobs Week 5:")
print(f"  Press Rating: {press}/10")
print(f"  Injury Concern: {injury}/5")
print(f"  Opponent Defense (LAR): {opp_def}/10")
print(f"  Vegas Sentiment: {vegas}/10")


################################################################################
TEST CASE 3: Josh Jacobs - Week 5, 2024
################################################################################

PRESS RATING: Josh Jacobs - Week 5

[Search Query 0]
Query: Josh Jacobs NFL week 5 2024 season

[Search Results 1]
  - Josh Jacobs Week 5 Outlook - Packers at Rams (2024) - Mick Ciallela (Fantrax) | FantasyPros
    URL: https://www.fantasypros.com/nfl/notes/384997/josh-jacobs-2024-week-5-outlook.php
  - Josh Jacobs - Green Bay Packers Running Back - ESPN
    URL: https://www.espn.com/nfl/player/_/id/4047365/josh-jacobs
  - Josh Jacobs Career Stats | NFL.com
    URL: https://www.nfl.com/players/josh-jacobs-x3315/stats/career
  - Josh Jacobs, Green Bay Packers, RB - Fantasy Football News, Stats - CBSSports.com
    URL: https://www.cbssports.com/nfl/players/2257876/josh-jacobs/fantasy/
  - Josh Jacobs Fantasy Football News, Rankings, Projections | Green Bay Packers | FantasyPros
    URL: 

## Verification Notes

After running the above cells, verify:

1. **Search queries are relevant** - Does Claude generate good search queries?
2. **Sources are credible** - Are the URLs from ESPN, NFL.com, Fantasy sites, Vegas sites?
3. **Reasoning is sound** - Does the explanation match the rating?
4. **Ratings make sense** - Do the numbers align with what you know about these games?

If everything looks good, you can proceed with running the full dataset in the main notebook!
